# Qwen3-TTS Voice Clone (Kaggle T4 Optimized)

This notebook provides a Gradio interface for voice cloning using Qwen3-TTS, optimized for Kaggle's T4 instances.

In [3]:
# Install prerequisites
!pip install -q -U qwen-tts
!pip install -q gradio soundfile

In [ ]:
import torch
import soundfile as sf
import gradio as gr
import os
import gc
import tempfile
import traceback
import warnings
from qwen_tts import Qwen3TTSModel

# Suppress warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

model = None

def load_model():
    global model
    if model is None:
        print("Loading Qwen3-TTS Base Model...")
        # For Voice Clone, we use the 1.7B-Base model
        model = Qwen3TTSModel.from_pretrained(
            "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
            device_map="cuda:0",
            dtype=dtype,
            # Kaggle T4 doesn't support flash_attention_2 efficiently, use sdpa instead
            attn_implementation="sdpa"
        )
        if device == "cuda":
            torch.cuda.empty_cache()
            gc.collect()
        print("Model loaded successfully!")
    return model

def generate_clone(text, ref_audio, ref_text, language, x_vector_only, progress=gr.Progress()):
    if not text:
        return None, "⚠️ Please enter text to generate!"
    if not ref_audio:
        return None, "⚠️ Please upload a reference audio for voice cloning!"
    if not ref_text and not x_vector_only:
        return None, "⚠️ Please enter the transcript of the reference audio (or enable 'X-Vector Only')."
        
    try:
        progress(0, desc="Loading model...")
        test_model = load_model()
        
        progress(0.4, desc="Generating audio... This may take a moment.")
        
        if x_vector_only:
            ref_text = None
            
        wavs, sr = test_model.generate_voice_clone(
            text=text,
            language="auto" if language == "Auto" else language,
            ref_audio=ref_audio,
            ref_text=ref_text,
            x_vector_only_mode=x_vector_only
        )
        
        progress(0.9, desc="Saving output...")
        output_path = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
        sf.write(output_path, wavs[0], sr)
        
        # Clear memory
        if device == "cuda":
            torch.cuda.empty_cache()
            gc.collect()
            
        return output_path, "✅ Generation successful!"
        
    except torch.cuda.OutOfMemoryError as e:
        if device == "cuda":
            torch.cuda.empty_cache()
            gc.collect()
        return None, f"❌ OUT OF MEMORY ERROR. Try shorter text, or click 'Clear VRAM'."
    except Exception as e:
        return None, f"❌ Error: {str(e)}\n\n{traceback.format_exc()}"

def clear_vram():
    global model
    if model is not None:
        del model
        model = None
    if device == "cuda":
        torch.cuda.empty_cache()
        gc.collect()
    return "✅ Model unloaded and VRAM cleared!"

# =============================================
# GRADIO INTERFACE WITH BRANDING
# =============================================

custom_css = """
.aiquest-header {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    padding: 20px;
    border-radius: 12px;
    margin-bottom: 16px;
    text-align: center;
    color: white;
}
.aiquest-header h1 {
    margin: 0 0 8px 0;
    font-size: 1.8em;
    color: white !important;
}
.aiquest-header p {
    margin: 4px 0;
    opacity: 0.95;
    color: white !important;
}
.social-buttons {
    display: flex;
    justify-content: center;
    gap: 12px;
    margin-top: 12px;
    flex-wrap: wrap;
}
.social-buttons a {
    display: inline-flex;
    align-items: center;
    gap: 6px;
    padding: 8px 18px;
    border-radius: 8px;
    text-decoration: none;
    font-weight: 600;
    font-size: 0.95em;
    transition: transform 0.2s, box-shadow 0.2s;
}
.social-buttons a:hover {
    transform: translateY(-2px);
    box-shadow: 0 4px 12px rgba(0,0,0,0.3);
}
.yt-btn {
    background: #FF0000;
    color: white !important;
}
.x-btn {
    background: #000000;
    color: white !important;
}
"""

with gr.Blocks(title="Qwen3-TTS Voice Clone by AIQUEST", theme=gr.themes.Soft(), css=custom_css) as demo:
    gr.HTML("""
    <div class=\"aiquest-header\">
        <h1>🎙️ Qwen3-TTS 1.7B Voice Cloning</h1>
        <p>1.7B Base Model • Optimized for Kaggle Free Tier (T4 GPU)</p>
        <p style=\"font-size:0.85em; opacity:0.8;\">Model by Qwen • Notebook by <b>AIQUEST</b></p>
    </div>
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            text_input = gr.Textbox(
                label="📝 Target Text", 
                placeholder="What do you want the cloned voice to say?", 
                lines=5,
                value="This is Qwen3-TTS Voice Clone running on Kaggle. It's really fast and accurate! Notebook by AIQUEST."
            )
            
            with gr.Group():
                gr.Markdown("### 🎤 Reference Voice to Clone")
                ref_audio = gr.Audio(
                    label="Reference Audio (~3-10s works best)", 
                    type="filepath", 
                    sources=["upload"]
                )
                ref_text = gr.Textbox(
                    label="Reference Transcript", 
                    placeholder="Enter the EXACT transcript of the reference audio.",
                    lines=2
                )
                x_vector_only = gr.Checkbox(
                    label="X-Vector Only Mode (Ignores Transcript, Lower Quality)", 
                    value=False
                )
            
            language = gr.Dropdown(
                choices=["Auto", "Chinese", "English", "Japanese", "Korean", "German", "French", "Russian", "Portuguese", "Spanish", "Italian"],
                value="Auto",
                label="Target Language"
            )
            
            with gr.Row():
                generate_btn = gr.Button("🎵 Generate Speech", variant="primary", size="lg", scale=3)
                clear_btn = gr.Button("🧹 Clear VRAM", variant="secondary", scale=1)
                
        with gr.Column(scale=1):
            audio_output = gr.Audio(label="🔊 Generated Audio", type="filepath")
            status_output = gr.Textbox(label="📊 Status", lines=16, interactive=False)
            
    generate_btn.click(
        fn=generate_clone,
        inputs=[text_input, ref_audio, ref_text, language, x_vector_only],
        outputs=[audio_output, status_output]
    )
    
    clear_btn.click(
        fn=clear_vram,
        inputs=[],
        outputs=[status_output]
    )

demo.launch(share=True, debug=True)


2026-04-01 03:33:30.429758: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775014410.665836      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775014410.730104      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775014411.237075      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775014411.237121      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775014411.237124      55 computation_placer.cc:177] computation placer alr


********
********
 
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://1d7d2ba422b1847b1d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loading Qwen3-TTS Base Model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Model loaded successfully!
